In [16]:
import pandas as pd
import duckdb as db
import pyodbc

In [17]:
conn = pyodbc.connect(
"DSN=producao;"
"UID=romilson;"
"PWD=Janber@1308"
)

In [20]:
con = db.connect()

In [21]:
df1 = pd.read_sql("select * from tbl_producao", conn)

C:\Users\romjs\AppData\Local\Temp\ipykernel_8016\3950015507.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df1 = pd.read_sql("select * from tbl_producao", conn)


In [25]:
con.register("df_oracle",df1)


In [26]:
df_oracle = con.execute("SELECT * FROM df_oracle").fetchdf()

In [ ]:
df2 = pd.read_csv("TBL_PRODUCAO_ACUMULADA.csv", sep=";")

# Padroniza nomes das colunas para evitar diferenças de caixa/espaco
if 'df_oracle' in globals():
    df_oracle.columns = [str(col).strip().lower() for col in df_oracle.columns]
df2.columns = [str(col).strip().lower() for col in df2.columns]

# Comparação dos tipos de dados entre os DataFrames
comparacao_dtypes = (
    df_oracle.dtypes.astype(str)
    .to_frame("df_oracle")
    .join(df2.dtypes.astype(str).to_frame("df_csv"), how="outer")
)
comparacao_dtypes["mesmo_tipo"] = comparacao_dtypes["df_oracle"] == comparacao_dtypes["df_csv"]

# Normaliza a chave de join para evitar divergencia de tipos
for col in ["proposta"]:
    if col in df_oracle.columns and col in df2.columns:
        df_oracle[col] = df_oracle[col].astype(str).str.strip()
        df2[col] = df2[col].astype(str).str.strip()

comparacao_dtypes


In [40]:
df2.head()

,Emp,Loja,Nome_Loja,UF_LOJA,UF_ILIAL,UF_PRODUCAO,PAIS,Filial,Nome_Filial,COORDENADORAS,...,CPF_CLIENTE,NOME_CLIENTE,Intervalo,VAL_TAC,SITUACAO_PG,Status,Msg,CONTRATO,TARIFA,UF_CLIENTE
0,1,258,GOV AC,AC,AC,AC,BRA,29,CREDIJIPA AC,ANA CARLA,...,21761884204,EDMILSON DA SILVA PAES,NaN,60,Normal,Liberadas/Pagas,NaN,483042129.0,"118,2",AC
1,1,258,GOV AC,AC,AC,AC,BRA,29,CREDIJIPA AC,ANA CARLA,...,56689560678,JOAO ARNALDO LEAL,NaN,60,Normal,Liberadas/Pagas,NaN,NaN,"1228,11",AC
2,1,1326,PREF RIO BRANCO,AC,AC,AC,BRA,12,CREDIJIPA AC,ANA CARLA,...,68692196215,NAJARA JANAYRA DE OLIVEIRA DA SILVA,TAB 1 - Z,30,Normal,Liberadas/Pagas,NaN,483077941.0,"905,78",AC
3,1,258,GOV AC,AC,AC,AC,BRA,29,CREDIJIPA AC,ANA CARLA,...,69558515949,LUIZ MIGUEL DE OLIVEIRA AGUIAR,TAB 7.2 - Z,60,Normal,Liberadas/Pagas,NaN,483093068.0,"639,23",AC
4,1,258,GOV AC,AC,AC,AC,BRA,29,CREDIJIPA AC,ANA CARLA,...,2693844215,DAVID ELIAS ABUGOCHE,TAB 7.2 - Z,0,Normal,Liberadas/Pagas,NaN,483107140.0,"553,68",AC


In [28]:
con.register("df_csv",df2)

In [ ]:
# Merge robusto em pandas, normalizando tipos antes do cruzamento
for col in ["proposta"]:
    if col in df_oracle.columns and col in df2.columns:
        df_oracle[col] = df_oracle[col].astype(str).str.strip()
        df2[col] = df2[col].astype(str).str.strip()

# Mantém apenas as colunas relevantes do CSV para o merge
csv_join = df2[["proposta", "nome_loja"]].drop_duplicates()

df_final = df_oracle.merge(csv_join, on="proposta", how="left")

df_final.head(10)

In [42]:
df_final.head(10)


,NOME_FILIAL,COORDENADORAS,GERENTES,VALOR_LIBERADO,DT_GERACAO_PG,DT_PG,DT_LANCAMENTO,DT_MOV,PROPOSTA,DATA_AMERICANA,Nome_Loja
0,VALOR PROMOTORA,ELAINE,AMANDA SANTOS,8000.0,2012-04-02,2012-04-03,2012-03-30,2012-03-30,8997520.0,2012-04-02,MPF DF
1,LEVYCRED,ELAINE,AMANDA SANTOS,8554.0,2012-04-02,2012-04-02,2012-03-30,2012-03-30,8997768.0,2012-04-02,MIN PLANEJAMENTO ORCAMENTO E GESTAO
2,VALOR PROMOTORA,ELAINE,AMANDA SANTOS,809683.0,2012-04-02,2012-04-02,2012-03-30,2012-03-30,8998273.0,2012-04-02,MIN PLANEJAMENTO ORCAMENTO E GESTAO
3,BR MERCANTIL,SARA,SOCORRO,208368.0,2012-04-02,2012-04-02,2012-04-02,2012-04-02,8998905.0,2012-04-02,PREF MACEIO
4,VALOR PROMOTORA,ELAINE,AMANDA SANTOS,251983.0,2012-04-03,2012-04-04,2012-04-02,2012-04-02,8998916.0,2012-04-03,MPF DF
5,BR MERCANTIL,SARA,SOCORRO,1357034.0,2012-04-02,2012-04-02,2012-04-02,2012-04-02,8998969.0,2012-04-02,PREF MACEIO
6,BR MERCANTIL,SARA,SOCORRO,28503.0,2012-04-02,2012-04-02,2012-04-02,2012-04-02,8999030.0,2012-04-02,PREF MACEIO
7,LEVYCRED,ELAINE,AMANDA SANTOS,4900.0,2012-04-03,2012-04-03,2012-04-02,2012-04-02,8999089.0,2012-04-03,INSS
8,BR MERCANTIL,SARA,SOCORRO,10439.0,2012-04-11,2012-04-11,2012-04-02,2012-04-02,8999148.0,2012-04-11,PREF MACEIO
9,BR MERCANTIL,SARA,SOCORRO,251369.0,2012-04-02,2012-04-03,2012-04-02,2012-04-02,8999179.0,2012-04-02,PREF MACEIO


In [47]:
df_filtro = df_final[df_final['Nome_Loja']=="MPF DF"]

In [46]:
df_filtro.head(10)

,NOME_FILIAL,COORDENADORAS,GERENTES,VALOR_LIBERADO,DT_GERACAO_PG,DT_PG,DT_LANCAMENTO,DT_MOV,PROPOSTA,DATA_AMERICANA,Nome_Loja
0,VALOR PROMOTORA,ELAINE,AMANDA SANTOS,8000.0,2012-04-02,2012-04-03,2012-03-30,2012-03-30,8997520.0,2012-04-02,MPF DF
4,VALOR PROMOTORA,ELAINE,AMANDA SANTOS,251983.0,2012-04-03,2012-04-04,2012-04-02,2012-04-02,8998916.0,2012-04-03,MPF DF
50,LEVYCRED,ELAINE,AMANDA SANTOS,2000.0,2012-04-04,2012-04-04,2012-03-30,2012-03-30,8998196.0,2012-04-04,MPF DF
51,LEVYCRED,ELAINE,AMANDA SANTOS,2000.0,2012-04-04,2012-04-04,2012-03-30,2012-03-30,8998099.0,2012-04-04,MPF DF
81,VALOR PROMOTORA,ELAINE,AMANDA SANTOS,15000.0,2012-04-09,2012-04-10,2012-04-03,2012-04-03,9000708.0,2012-04-09,MPF DF
82,VALOR PROMOTORA,ELAINE,AMANDA SANTOS,7000.0,2012-04-09,2012-04-10,2012-04-03,2012-04-03,9000616.0,2012-04-09,MPF DF
83,LEVYCRED,ELAINE,AMANDA SANTOS,16000.0,2012-04-09,2012-04-10,2012-04-03,2012-04-03,9000788.0,2012-04-09,MPF DF
97,VALOR PROMOTORA,ELAINE,AMANDA SANTOS,202009.0,2012-04-10,2012-04-11,2012-04-09,2012-04-09,9005283.0,2012-04-10,MPF DF
98,VALOR PROMOTORA,ELAINE,AMANDA SANTOS,1402353.0,2012-04-10,2012-04-11,2012-04-04,2012-04-04,9002378.0,2012-04-10,MPF DF
99,VALOR PROMOTORA,ELAINE,AMANDA SANTOS,373319.0,2012-04-10,2012-04-11,2012-04-09,2012-04-09,9005131.0,2012-04-10,MPF DF
